## Academic Success Classification

This project predicts a student's academic outcome, based on demographic, socio-economic, and academic performance data. It is a 3-class classification problem: the student either dropped out, is still enrolled, or graduated.

## Approach
1. Load the data and explore the target balance
2. Train a baseline multi-class classifier and evaluate it with accuracy on a validation split
3. Generate predictions and create the submission file
4. Submit to Kaggle and record the score


In [1]:
import os
from getpass import getpass

os.environ["KAGGLE_API_TOKEN"] = getpass("Paste your Kaggle API token and press Enter: ")

!pip install -q -U kaggle
!kaggle competitions download -c playground-series-s4e6
!unzip -oq playground-series-s4e6.zip

Paste your Kaggle API token and press Enter: ··········
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 2.0 MB/s eta 0:00:00
100% 3.07M/3.07M [00:00<00:00, 131MB/s]



In [2]:
import pandas as pd

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

In [3]:
print(train.shape, test.shape)

(76518, 38) (51012, 37)


In [4]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76518 entries, 0 to 76517
Data columns (total 38 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   id                                              76518 non-null  int64  
 1   Marital status                                  76518 non-null  int64  
 2   Application mode                                76518 non-null  int64  
 3   Application order                               76518 non-null  int64  
 4   Course                                          76518 non-null  int64  
 5   Daytime/evening attendance                      76518 non-null  int64  
 6   Previous qualification                          76518 non-null  int64  
 7   Previous qualification (grade)                  76518 non-null  float64
 8   Nacionality                                     76518 non-null  int64  
 9   Mother's qualification                 

In [5]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51012 entries, 0 to 51011
Data columns (total 37 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   id                                              51012 non-null  int64  
 1   Marital status                                  51012 non-null  int64  
 2   Application mode                                51012 non-null  int64  
 3   Application order                               51012 non-null  int64  
 4   Course                                          51012 non-null  int64  
 5   Daytime/evening attendance                      51012 non-null  int64  
 6   Previous qualification                          51012 non-null  int64  
 7   Previous qualification (grade)                  51012 non-null  float64
 8   Nacionality                                     51012 non-null  int64  
 9   Mother's qualification                 

In [6]:
train.sample(1)

,id,Marital status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
2544,2544,1,17,4,9500,1,1,137.0,1,37,...,0,8,9,8,15.622222,0,13.9,-0.3,0.79,Graduate


In [7]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51012 entries, 0 to 51011
Data columns (total 37 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   id                                              51012 non-null  int64  
 1   Marital status                                  51012 non-null  int64  
 2   Application mode                                51012 non-null  int64  
 3   Application order                               51012 non-null  int64  
 4   Course                                          51012 non-null  int64  
 5   Daytime/evening attendance                      51012 non-null  int64  
 6   Previous qualification                          51012 non-null  int64  
 7   Previous qualification (grade)                  51012 non-null  float64
 8   Nacionality                                     51012 non-null  int64  
 9   Mother's qualification                 

In [8]:
train["Target"].value_counts()

,count
Target,
Graduate,36282
Dropout,25296
Enrolled,14940


In [9]:
train.dtypes.value_counts()

,count
int64,30
float64,7
object,1


In [10]:
train.select_dtypes(include="object").columns.tolist()

['Target']

In [11]:
x = train.drop(columns=["id", "Target"])
y = train["Target"]

In [12]:
print(x.shape, y.shape)

(76518, 36) (76518,)


In [13]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=300, random_state=42)
model.fit(x_train, y_train)

pred = model.predict(x_val)
print("Validation accuracy:", round(accuracy_score(y_val, pred), 4))

Validation accuracy: 0.8283


In [14]:
model.fit(x, y)

x_test = test.drop(columns=["id"])
pred_test = model.predict(x_test)

pd.DataFrame({"id": test["id"], "Target": pred_test}).to_csv("submission.csv", index=False)

In [15]:
!kaggle competitions submit -c playground-series-s4e6 -f submission.csv -m "RandomForest baseline"

100% 759k/759k [00:00<00:00, 1.21MB/s]
99 submissions remaining today.
Successfully submitted to Classification with an Academic Success Dataset

In [16]:
model = RandomForestClassifier(n_estimators=150, max_depth=10, min_samples_leaf=5, random_state=42)
model.fit(x, y)

defaults = {c: float(x[c].median()) for c in x.columns}

import pickle
pickle.dump(model, open("academic_model.pkl", "wb"))
pickle.dump({"feature_cols": list(x.columns), "defaults": defaults}, open("academic_meta.pkl", "wb"))